<a href="https://colab.research.google.com/github/hyui02/ESAA_assignment/blob/main/ESSA_WEEK12_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 수상작 리뷰: MLB Player Digital Engagement Forecasting (1위)

- 리뷰 노트북: [MLB_final_1](https://www.kaggle.com/code/ph0921/mlb-final-1) (1위 솔루션)
- 대회: [MLB Player Digital Engagement Forecasting (Kaggle, 2021)](https://www.kaggle.com/competitions/mlb-player-digital-engagement-forecasting)
- 한 줄 요약: 선수별 과거 96일 시퀀스(경기 기록과 과거 타깃값)를 GRU에 넣어 다음 날의 디지털 참여도 4종을 예측하고, 에폭별 체크포인트 중 검증 손실이 낮은 6개를 평균하여 제출하였다.

---

# **1. 데이터 설명**

## 1-1. 대회 과제
- 2021 시즌에 활동 중인 MLB 선수들을 대상으로 **다음 날의 디지털 참여도(engagement)** 네 가지(`target1` ~ `target4`)를 예측하는 시계열 회귀 문제이다.
- 날짜 d의 데이터가 주어지면 d+1일의 참여도를 예측한다.
- 네 타깃은 각각 **0~100 척도의 일별 디지털 참여도 지수**이며, 각 지표의 구체적인 의미는 공개되지 않았다.
- 평가지표는 **MCMAE(Mean Column-wise Mean Absolute Error)** 이다. 네 타깃 각각의 MAE를 구한 뒤 그 네 값을 평균한다.
- 예측 키는 `date_playerId`(예: `20210501_642727`)이며, 이 조합별로 네 타깃 값을 제출한다.

## 1-2. 대회 진행 방식
- 모델이 미래 데이터를 미리 보지 못하도록 **`mlb` 시계열 모듈**을 통해서만 제출한다. 모듈이 하루치 테스트 데이터(`test_df`)와 제출 양식(`sample_prediction_df`)을 차례로 넘겨주면, 그날의 예측을 `env.predict()`로 제출한다.
- 테스트 데이터는 `train.csv`와 같은 형식이지만 **타깃값이 들어 있지 않다.**
- 대회는 두 단계로 진행되었다.
    - **학습 단계(2021.06.10 ~ 07.31)**: 공개 리더보드의 테스트 구간은 **2021년 5월** 데이터이다.
    - **평가 단계(2021.08 ~ 09.15)**: 최종 제출 노트북 2개를 **약 한 달간의 미래 시즌 데이터**에 다시 실행하여 최종 순위를 정한다. 이때 학습 데이터는 2021년 7월 31일까지로 갱신된다.

## 1-3. 파일 구성
| 구분 | 파일 | 내용 |
|---|---|---|
| 일별 데이터 | `train.csv` | 2018년 1월 1일부터의 일별 데이터이다. 한 행이 하루이며, 각 열에 그날의 하위 테이블이 **JSON 문자열로 중첩**되어 있다. |
| 예시 | `example_test.csv`, `example_sample_submission.csv` | 테스트 데이터와 제출 파일의 형식 예시 |
| 정적 데이터 | `players.csv` | 선수 기본 정보(생년월일, 데뷔일, 키·몸무게, 주 포지션 등)와 예측 대상 여부(`playerForTestSetAndFuturePreds`) |
| | `teams.csv` | 구단 정보(리그, 지구, 홈구장 등) |
| | `seasons.csv` | 시즌별 정규시즌·올스타전·포스트시즌 일정 |
| | `awards.csv` | 2018년 이전 수상 이력 |

- `train.csv`에는 2018년 이후 한 번이라도 활동한 선수가 모두 들어 있지만, 채점은 **2021 시즌 활동 선수**에 대해서만 이루어진다.

## 1-4. 중첩 테이블과 노트북의 사용 여부
| 중첩 테이블 | 내용 | 사용 |
|---|---|---|
| `nextDayPlayerEngagement` | **타깃.** `engagementMetricsDate`, `playerId`, `target1~4` | ○ |
| `playerBoxScores` | 선수별 경기 기록(타격·투구·수비 지표, `home`, `positionCode` 등) | ○ |
| `games` | 경기 정보(`gamePk`, 경기 유형, 홈·원정 팀, `homeScore`, `awayScore`, 승률 등) | ○ |
| `rosters` | 날짜별 선수 소속 팀(`teamId`)과 로스터 상태(`statusCode`, `status`) | ○ |
| `playerTwitterFollowers` | 선수 트위터 팔로워 수(`numberOfFollowers`) | ○ |
| `teamBoxScores` | 팀별 경기 기록 | × |
| `transactions` | 트레이드 등 구단 간 선수 이동 | × |
| `standings` | 팀 순위·승패·게임차 | × |
| `awards` | 해당 날짜의 수상 내역 | × |
| `events` | 투구 단위 경기 이벤트(구속, 회전수, 타구 속도 등) | × |
| `teamTwitterFollowers` | 구단 트위터 팔로워 수 | × |

- 이 노트북은 11개 중첩 테이블 중 **타깃을 포함한 5개만** 사용하였다.

## 1-5. 데이터를 다룰 때 알아 둘 특징
- **날짜 정렬**: `engagementMetricsDate`는 미국 태평양 시간 기준이며, **전날의** 경기·로스터·기록과 짝을 이룬다.
- **결측과 0의 의미 차이**: 0은 **기회가 있었으나 하지 못한 경우**, 결측(null)은 **기회 자체가 없었던 경우**이다. 예를 들어 그날 등판하지 않은 투수의 완봉 여부는 0이 아니라 결측이다.
- **트위터 팔로워 수 수집 주기**: 선수 팔로워 수는 **매월 1일**에만 수집되며, 계정이 없는 선수 등은 빠져 있다.
- **점수 기준 시점**: 대부분의 경기 상태 컬럼은 이벤트 이후 값이지만, 홈·원정 점수는 이벤트 이전 값이다.

---

# **2. 코드 흐름**

1. **환경 설정**
    - 라이브러리를 불러오고, 실행 시작 시각(`global_time`)과 시간 예산(`max_sec` = 5시간)을 기록한다.
2. **데이터 불러오기**
    - `train_updated.csv`에서 사용할 5개 열만 읽고, 중첩 JSON을 `unpack_data`(joblib 병렬)로 풀어 테이블별 데이터프레임으로 만든다.
3. **병합 및 전처리(`merge_df`)**
    - `playerBoxScores`, `games`, `rosters`, `playerTwitterFollowers`를 날짜와 선수 기준으로 병합한다.
    - 파생변수 `diff`(점수 차), `num_games`(그날 경기 비율), `month`를 만들고, 미리 정한 표준편차(`feat_std`)로 수치형 변수를 나눈다.
4. **리그 평균 타깃(`agg_tars_1~4`) 생성**
    - 날짜별 모든 선수의 타깃 평균을 변수로 추가한다.
5. **범주형 인코딩**
    - 고정 매핑 사전으로 정수 변환 후, Dataset에서 one-hot으로 바꾼다.
6. **Dataset 구성(`MLBdata`)**
    - 선수별 96일 윈도우를 무작위로 뽑고, 과거 타깃을 입력에 붙이되 최근 1~32일의 타깃은 가린다.
7. **모델(`MLB_Model`)**
    - Linear → ResBlock 2개 → GRU → MLP → ReLU 구조이다.
8. **학습(`Trainer`)**
    - L1 Loss, Adam, 지수적 학습률 감소를 사용하고 2021년 5월로 검증한다.
    - 매 에폭 가중치를 저장한 뒤 검증 손실이 낮은 6개를 고른다.
9. **추론 및 제출(`Evaluator`)**
    - 시계열 모듈에서 하루치 데이터를 받을 때마다 기존 이력에 이어 붙이고, 6개 모델의 예측을 평균한 뒤 0~100으로 잘라 제출한다.

---

# **3. 주요 코드 분석**

- 아래 코드 셀은 원본 노트북의 핵심 로직을 그대로 옮기되, 대회 데이터 없이도 에러 없이 실행되도록 **작은 예시 데이터**를 넣은 것이다.
- 첫 코드 셀(라이브러리 불러오기)을 먼저 실행한 뒤 위에서부터 차례로 실행한다.

In [1]:
import time
import random
from datetime import timedelta

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F

**① 타깃 날짜를 하루 당기는 병합**

In [2]:
# 예시 데이터: 선수 2명, 타깃 날짜 2일
engagement = pd.DataFrame({
    'engagementMetricsDate': ['2021-05-02', '2021-05-02', '2021-05-03', '2021-05-03'],
    'playerId': [1, 2, 1, 2],
    'target1': [3.1, 0.5, 2.4, 0.7],
})
box_scores = pd.DataFrame({'gameDate': ['2021-05-01', '2021-05-02'], 'gamePk': [100, 200],
                           'playerId': [1, 1], 'teamId': [10, 10], 'hits': [2, 0]})
games = pd.DataFrame({'gameDate': ['2021-05-01', '2021-05-02'], 'gamePk': [100, 200],
                      'homeScore': [5, 2], 'awayScore': [3, 4]})
rosters = pd.DataFrame({'gameDate': ['2021-05-01'] * 2 + ['2021-05-02'] * 2,
                        'playerId': [1, 2, 1, 2], 'teamId': [10, 20, 10, 20],
                        'statusCode': ['A', 'A', 'A', 'D10']})
for df in [box_scores, games, rosters]:
    df['gameDate'] = pd.to_datetime(df['gameDate'])

def merge_df(test_df, infer=True):
    test_df = test_df.copy()
    test_df['gameDate'] = pd.to_datetime(test_df['engagementMetricsDate'])
    if not infer:
        test_df['gameDate'] = test_df['gameDate'] - timedelta(days=1)   # 타깃 날짜를 하루 당김
    df_train = box_scores.merge(games, how='outer', on=['gameDate', 'gamePk'])
    df_train['diff'] = (df_train['homeScore'] - df_train['awayScore']).fillna(0).values
    df_train = df_train.merge(rosters, how='outer', on=['gameDate', 'teamId', 'playerId'])
    df_train = df_train.groupby(['gameDate', 'playerId']).head(1)
    return test_df.merge(df_train, how='left', on=['gameDate', 'playerId'])

merge_df(engagement, infer=False)[['engagementMetricsDate', 'gameDate', 'playerId', 'hits', 'diff', 'statusCode', 'target1']]

,engagementMetricsDate,gameDate,playerId,hits,diff,statusCode,target1
0,2021-05-02,2021-05-01,1,2.0,2.0,A,3.1
1,2021-05-02,2021-05-01,2,NaN,NaN,A,0.5
2,2021-05-03,2021-05-02,1,0.0,-2.0,A,2.4
3,2021-05-03,2021-05-02,2,NaN,NaN,D10,0.7


- 공식 설명대로 `engagementMetricsDate`는 전날의 경기·로스터와 짝을 이루므로, 학습 시에는 타깃 날짜에서 하루를 빼 "전날 경기 기록 → 다음 날 참여도"가 한 행에 오도록 맞춘다.
- 경기 기록·경기 정보·로스터·팔로워 수를 모두 outer 병합한 뒤, 마지막에 타깃 테이블 기준으로 left 병합한다. 그 결과 경기를 뛰지 않은 날의 선수도 행이 유지되고, 경기 기록은 결측으로 남는다.
- 학습과 추론이 **같은 `merge_df` 함수**를 공유하므로 두 단계의 전처리가 어긋나지 않는다.

**② 파생변수와 스케일링**

In [3]:
df = pd.DataFrame({
    'gameDate': ['2021-05-01'] * 3 + ['2021-05-02'] * 3,
    'home': [1, 0, np.nan, np.nan, np.nan, 1],
    'hits': [2, 1, np.nan, np.nan, np.nan, 3],
})

def add_agg_feats(df, infer=True):
    f = 0.78 if infer else 1
    df['num_games'] = f * df.groupby('gameDate')['home'].transform(lambda x: 1 - (x != x).mean()).fillna(0)
    return df

feat_std = {'num_games': 0.2, 'hits': 0.8583428342850036}

df = add_agg_feats(df, infer=False)
for f, s in feat_std.items():
    df[f] = df[f].values / s

df['hits_input'] = df['hits'].fillna(-1)   # Dataset에서처럼 결측은 -1로 채움
df

,gameDate,home,hits,num_games,hits_input
0,2021-05-01,1.0,2.330071,3.333333,2.330071
1,2021-05-01,0.0,1.165036,3.333333,1.165036
2,2021-05-01,NaN,NaN,3.333333,-1.000000
3,2021-05-02,NaN,NaN,1.666667,-1.000000
4,2021-05-02,NaN,NaN,1.666667,-1.000000
5,2021-05-02,1.0,3.495107,1.666667,3.495107


- `num_games`는 날짜별로 `home` 값이 결측이 아닌 비율로, **그날 경기가 얼마나 열렸는지**를 나타낸다. `x != x`는 NaN을 판별하는 관용 표현이다(NaN은 자기 자신과 같지 않다).
- 추론 시에는 `f = 0.78`을 곱하며, 이 값을 정한 근거는 노트북에 제시되어 있지 않다.
- 수치형 변수는 `StandardScaler` 대신 **미리 계산해 둔 표준편차 사전(`feat_std`)** 으로 나눈다. 값이 매우 작은 변수는 하한 0.2를 두었다. 추론 시 데이터가 하루치뿐이어도 학습 때와 같은 스케일을 적용할 수 있다.
- Dataset에서 결측은 0이 아닌 **-1로 채운다(`fillna(-1)`)**. 공식 설명의 "0은 기회가 있었으나 하지 못함, 결측은 기회가 없었음"이라는 구분이 입력에서도 유지된다.

**③ 리그 전체 평균 타깃(`agg_tars`)**

In [4]:
tar_cols = ['target1', 'target2', 'target3', 'target4']
df_train = pd.DataFrame({
    'gameDate': ['2021-05-01'] * 3 + ['2021-05-02'] * 3,
    'playerId': [1, 2, 3, 1, 2, 3],
    'target1': [10.0, 2.0, 0.0, 30.0, 6.0, 3.0],
    'target2': [5.0, 1.0, 0.0, 9.0, 3.0, 0.0],
    'target3': [1.0, 0.0, 0.0, 2.0, 1.0, 0.0],
    'target4': [8.0, 2.0, 1.0, 20.0, 4.0, 2.0],
})

for i in range(1, 5):
    df_train[f'agg_tars_{i}'] = df_train.groupby('gameDate')[tar_cols[i-1]].transform('mean').fillna(0)

df_train

,gameDate,playerId,target1,target2,target3,target4,agg_tars_1,agg_tars_2,agg_tars_3,agg_tars_4
0,2021-05-01,1,10.0,5.0,1.0,8.0,4.0,2.0,0.333333,3.666667
1,2021-05-01,2,2.0,1.0,0.0,2.0,4.0,2.0,0.333333,3.666667
2,2021-05-01,3,0.0,0.0,0.0,1.0,4.0,2.0,0.333333,3.666667
3,2021-05-02,1,30.0,9.0,2.0,20.0,13.0,4.0,1.000000,8.666667
4,2021-05-02,2,6.0,3.0,1.0,4.0,13.0,4.0,1.000000,8.666667
5,2021-05-02,3,3.0,0.0,0.0,2.0,13.0,4.0,1.000000,8.666667


- 날짜별 **모든 선수의 타깃 평균**으로, 리그 전체 관심도의 흐름을 모델에 알려 준다.
- 개별 선수 값과 리그 평균을 함께 입력하므로, 모델이 "이 선수가 평소보다 주목받는가"와 "오늘 야구 전체가 주목받는가"를 나누어 학습할 수 있다.

**④ 과거 타깃을 입력으로 쓰고 예측 공백을 재현하는 Dataset (핵심)**

In [5]:
NUM = 32      # 출력(예측) 구간 길이
NUM_1 = 3     # 입력 구간 = NUM * NUM_1 = 96일
num = NUM_1 * NUM

# 예시 데이터: 한 선수의 150일 이력(경기 기록 5개, 타깃 4개, 리그 평균 타깃 4개)
n_days = 150
ind = list(range(n_days))
all_feats = torch.randn(n_days, 5)
all_tars = torch.rand(n_days, 4) * 10
all_agg_tars = torch.rand(n_days, 4) * 5

idx = random.randint(NUM * NUM_1, len(ind) - 1)   # 선수 이력 중 기준 시점
idx_1 = random.randint(1, min(len(ind), NUM))     # 타깃을 모르는 최근 일수(1~32일)

ind_1 = np.array(ind)[max(0, idx + 1 - num):idx + 1]   # 입력 96일 전체
ind_2 = ind[max(0, idx + 1 - num):idx + 1 - idx_1]     # 그중 타깃을 아는 구간

feats = all_feats[ind_1]
tar_feats = all_tars[ind_2]
tar_feats_1 = torch.cat([tar_feats.mean(0), tar_feats.median(0)[0]], -1)   # 아는 구간의 평균·중앙값
tar_feats_1 = tar_feats_1[None, ...] + 0 * torch.zeros(len(feats), 1)

tar_feats = torch.cat([all_tars[ind_2], 0 * tar_feats[:, :1], all_agg_tars[ind_2]], -1)
tar_pad = torch.zeros((len(feats) - len(tar_feats), 9))
tar_pad[:, 4] = 1                                   # 타깃을 모르는 구간 표시
tar_feats = torch.cat([torch.log(1 + tar_feats), tar_pad], 0)
x = torch.cat([tar_feats, feats, tar_feats_1], -1)

tars = all_tars[ind_1[-NUM:]].clone()
tars[:-idx_1, :] = -1                               # 가려진 구간만 정답으로 남김

print('가려진 최근 일수 idx_1 :', idx_1)
print('입력 크기              :', tuple(x.shape))
print('플래그=1인 시점 수     :', int(x[:, 4].sum()))
print('정답이 남은 시점 수    :', int((tars[:, 0] != -1).sum()))

가려진 최근 일수 idx_1 : 9
입력 크기              : (96, 22)
플래그=1인 시점 수     : 9
정답이 남은 시점 수    : 9


- **선수 단위 시퀀스**: 선수별 행 인덱스를 모아 두고, 기준 시점에서 과거 96일을 잘라 한 샘플로 만든다.
- **과거 타깃을 입력으로 사용**: 경기 기록뿐 아니라 이전 날짜의 `target1~4`(log1p 변환)와 `agg_tars`를 입력에 붙인다. 알고 있는 구간 타깃의 평균·중앙값도 모든 시점에 복제해 붙인다.
- **예측 공백 재현**: 실제 평가에서는 학습 데이터 이후 약 한 달 동안 타깃을 알 수 없다. 이를 학습에서 재현하기 위해 샘플마다 최근 `idx_1`일(1~32일, 무작위)의 타깃을 가리고 0으로 채운다. 채운 위치에는 **플래그 컬럼(인덱스 4)에 1**을 넣어 "모르는 값"임을 모델에 알린다.
- **손실 마스킹**: 정답은 가려진 최근 `idx_1`일만 남기고 나머지는 -1로 채운다. 학습 시 `t != -1`인 위치만 손실을 계산하므로, 모델은 **모르는 구간을 예측하는 능력**만 학습한다.
- **무작위 윈도우**: 한 에폭은 `8 * 데이터 수 / 배치 크기`개 샘플이며, 매번 `random.choice`로 새 윈도우를 뽑는다. 같은 데이터에서 다양한 학습 샘플을 만들어 낸다.

**⑤ 모델 구조**

In [6]:
NUM_F, NUM_C, UNITS = 30, 60, 128   # 예시 입력 차원

class ResBlock(nn.Module):
    def __init__(self, in_units, units=None):
        super().__init__()
        if units is None:
            units = 2 * in_units
        self.hidden = nn.Linear(in_units, units)
        self.out = nn.Linear(units, in_units)

    def forward(self, x0):
        x = F.leaky_relu(self.hidden(x0))
        return F.leaky_relu(x0 + self.out(x))

class MLB_Model(nn.Module):
    def __init__(self, units):
        super().__init__()
        num_c = 24
        self.in_lay = nn.Linear(NUM_F, NUM_F)
        self.in_lay_c = nn.Linear(NUM_C, num_c)
        self.merge = nn.Linear(NUM_F + num_c, units)
        self.process = nn.Sequential(*[ResBlock(units) for _ in range(2)])
        self.drop_0 = nn.Dropout(.12)
        self.drop_1 = nn.Dropout(.16)
        self.rnn = nn.GRU(units, units, batch_first=True)
        self.out_1 = nn.Linear(2 * units, 2 * units)
        self.out_2 = nn.Linear(2 * units, 2 * units)
        self.out_3 = nn.Linear(2 * units, 4)

    def forward(self, x, x_c):
        x_0 = F.leaky_relu(self.in_lay(x))                          # 수치형(타깃 + 경기 기록)
        x_1 = F.leaky_relu(self.in_lay_c(x_c))                      # 범주형 one-hot → 24차원
        x = F.leaky_relu(self.merge(torch.cat([x_0, x_1], -1)))     # → 128차원
        x = self.process(self.drop_0(x))                            # ResBlock × 2
        x_0 = self.rnn(x)[0]                                        # GRU
        x = torch.cat([x[:, -NUM:, :], x_0[:, -NUM:, :]], -1)       # 마지막 32일
        x = self.drop_1(x)
        x = F.leaky_relu(self.out_1(x))
        x = F.leaky_relu(self.out_2(x))
        return F.relu(self.out_3(x))                                # 타깃 4개, 음수 방지

model = MLB_Model(UNITS)
out = model(torch.randn(8, 96, NUM_F), torch.randn(8, 96, NUM_C))   # 배치 8, 96일
print('출력 크기:', tuple(out.shape))   # (배치, 마지막 32일, 타깃 4개)

출력 크기: (8, 32, 4)


- **시점별 특징 추출 → GRU로 시간 흐름 요약 → MLP 출력**의 단순한 구조이다.
- GRU 출력과 GRU에 들어가기 전의 시점별 특징을 이어 붙여(skip connection), 시간 맥락과 그날의 개별 정보를 함께 사용한다.
- 마지막 활성화를 ReLU로 두어 예측값이 음수가 되지 않게 하였다.
- 범주형 변수(`positionCode`, `teamId`, `statusCode`, `month`)는 `torch.eye(num)[...]`로 one-hot을 만든 뒤 Linear 층으로 24차원으로 압축한다.
- WaveNet 방식의 `Wave_Block`도 정의되어 있으나 주석 처리되어 실제로는 GRU만 사용하였다.

**⑥ 학습 전략**

In [7]:
epochs, lr, end_lr = 30, 1.6e-3, 8e-4
fact = (end_lr / lr) ** (1 / (1 + epochs))       # 에폭마다 곱할 학습률 감소 비율
opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=6e-6)

global_time = time.time()
max_sec = 5 * 3600                               # 시간 예산 5시간

for step in range(3):
    x = torch.randn(8, 96, NUM_F)
    x_c = torch.randn(8, 96, NUM_C)
    t = torch.rand(8, NUM, 4) * 10
    t[:, :-5, :] = -1                            # 가려진 최근 5일만 정답으로 남김

    opt.zero_grad()
    p = model(x, x_c)
    mask = t != -1
    loss = torch.abs(p[mask] - t[mask]).mean()   # L1 Loss = MAE
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1., norm_type='inf')
    opt.step()
    print(f'step {step} loss: {loss.item():.4f}')

    if time.time() - global_time > max_sec:      # 5시간 초과 시 중단
        break

lr *= fact
for g in opt.param_groups:
    g['lr'] = lr
print('학습률 감소 비율:', round(fact, 4), '/ 다음 에폭 학습률:', lr)

step 0 loss: 5.0877
step 1 loss: 4.6972
step 2 loss: 4.9896
학습률 감소 비율: 0.9779 / 다음 에폭 학습률: 0.0015646216581366925


- 손실함수를 **평가지표(MAE)와 같은 L1 Loss**로 두었다.
- 학습률은 1.6e-3에서 8e-4까지 에폭마다 같은 비율로 줄인다.
- 실행 시작 후 5시간이 지나면 학습을 멈춘다. Kaggle 노트북 실행 시간 제한에 맞춘 장치이다.
- 매 에폭 가중치를 저장하고, **검증 손실이 낮은 6개 체크포인트**를 골라 추론 시 평균한다. 추가 학습 없이 예측을 안정시키는 방법이다.
- 검증 구간은 **2021년 5월**로, 학습 단계 공개 리더보드의 테스트 구간과 같다.

**⑦ 실시간 추론**

In [8]:
num = NUM_1 * NUM   # 96

# 예시: 학습 데이터에 200일, 테스트 기간 5일이 더 이어 붙은 선수
ind_0 = list(range(200))          # 학습 데이터 이력(타깃을 아는 구간)
ind = list(range(205))            # 테스트 날짜까지 이어 붙인 전체 이력

delta = len(ind) - len(ind_0)     # 학습 데이터 이후 지난 일수
ind_1 = ind[-num:]                # 입력: 최근 96일 경기 기록
ind_2 = ind_0[delta - num:]       # 타깃: 학습 데이터에 있는 부분만

print('delta                :', delta)
print('입력 구간            :', ind_1[0], '~', ind_1[-1], f'({len(ind_1)}일)')
print('타깃을 아는 구간     :', ind_2[0], '~', ind_2[-1], f'({len(ind_2)}일)')
print('타깃을 모르는 일수   :', len(ind_1) - len(ind_2))

preds = np.random.rand(3, 4) * 120 - 10
print(np.clip(preds, 0, 100))   # 최종 예측을 0~100 범위로 자름

delta                : 5
입력 구간            : 109 ~ 204 (96일)
타깃을 아는 구간     : 109 ~ 199 (91일)
타깃을 모르는 일수   : 5
[[ 0.         24.75487429 26.31525286 23.09481689]
 [21.44586831 35.07931195 93.56539536  0.        ]
 [91.12929423 47.97167432 20.38580178 13.56401549]]


- 학습 데이터 중 마지막 100만 행만 이력으로 유지한다.
- 시계열 모듈에서 하루치 데이터가 들어올 때마다 같은 `merge_df`와 인코딩을 거쳐 이력에 이어 붙인다.
- 테스트 기간의 타깃은 주어지지 않으므로, 경기 기록은 최신까지 쓰되 **타깃은 학습 데이터에 있는 구간까지만** 사용한다. 날이 지날수록 `delta`가 커지며, 이는 학습 때 `idx_1`로 무작위 공백을 연습시킨 것과 대응한다.
- 그날 비어 있는 테이블은 컬럼 구조만 갖춘 결측 데이터프레임으로 대체하여 오류 없이 처리한다.
- 최종 예측은 6개 모델 평균 후 `np.clip(..., 0, 100)`으로 타깃 범위 안에 맞춘다.

---

# **4. 새로 알게 된 점 / 배운 점**

**배운 점**
- **평가 상황을 학습에서 재현하는 설계**: 실제 평가에서는 최근 약 한 달의 타깃을 알 수 없다. 이 노트북은 학습 샘플마다 공백 길이(1~32일)를 무작위로 정해 타깃을 가리고, 가려진 구간만 손실로 학습시켰다. 모델 선택보다 **학습과 추론의 조건을 맞추는 일**이 중요하다는 점을 배웠다.
- **과거 타깃을 입력으로 쓰는 방법**: 참여도처럼 날마다 이어지는 시계열에서는 과거 타깃과 그 평균·중앙값이 강한 입력이 된다. 모르는 구간을 0으로만 채우면 실제 0과 구분되지 않으므로 **별도의 마스크 플래그 컬럼**을 함께 넣는 기법을 알게 되었다.
- **리그 평균이라는 맥락 변수**: 개별 값과 전체 흐름을 나누어 보게 하는 간단한 파생변수이다.
- **손실함수를 평가지표에 맞추기**: MAE로 채점하는 대회에서 L1 Loss를 직접 최적화하였다.
- **체크포인트 앙상블**: 모델을 여러 번 학습하지 않고도 상위 에폭의 가중치를 평균하여 예측을 안정시킬 수 있다.
- **시계열 코드 대회 대응**: 실행 시간 관리, 이력 데이터 축소, 비어 있는 테이블 대체 등 대회 환경에 맞춘 처리를 볼 수 있었다.
- **데이터 설명을 코드에 반영하기**: "결측은 기회 없음, 0은 기회 있었으나 실패"라는 공식 설명이 `fillna(-1)`로, "참여도는 전날 경기와 짝을 이룸"이라는 설명이 날짜를 하루 빼는 병합으로 이어졌다.

**아쉬운 점과 개선 방향**
- **검증 데이터 누수 가능성**: 2021년 5월을 검증으로 뺄 때 기준 행만 제외하고, 실제 윈도우는 선수 전체 이력에서 무작위로 잡는다. 따라서 5월 데이터가 학습 윈도우에 들어갈 수 있어 검증 손실이 실제보다 낮게 나올 수 있다.
- **마지막 에폭 검증값 오류**: 26번째 에폭의 검증 손실이 0.137로 다른 에폭(약 0.70)보다 비정상적으로 낮다. 30에폭을 지정했으나 26에폭에서 멈춘 것은 5시간 초과 조건이 작동했기 때문이며, 이 조건은 검증 루프에도 똑같이 적용되어 26에폭 검증은 일부 배치만 평가되었다. 그 결과 해당 체크포인트가 상위 6개의 첫 번째로 선택되었다. 시간 초과 시의 검증 결과는 제외해야 한다.
- **트위터 팔로워 수 대부분 결측**: 팔로워 수는 매월 1일에만 수집되는데, 노트북은 날짜가 정확히 같을 때만 병합한다. 따라서 매월 1일을 제외한 날에는 팔로워 수가 결측(-1)이 된다. 선수별로 직전 값을 앞으로 채우면(forward fill) 매일 쓸 수 있는 인기 지표가 된다.
- **고정된 상수**: 범주형 매핑 사전, `feat_std`, 추론 시 보정 계수 0.78이 근거 없이 코드에 고정되어 있어 재현과 해석이 어렵다.
- **추론 속도**: 선수 한 명씩 반복하며 예측하므로 배치 단위 추론보다 느리다.

**내 프로젝트에 적용할 점**
- 시계열 예측에서는 **추론 시점에 무엇을 알고 무엇을 모르는지**를 먼저 정하고, 그 조건을 학습 샘플을 만들 때 재현한다.
- 결측과 실제 0을 구분하기 위해 마스크 플래그를 함께 입력한다.
- 시간 기준으로 검증할 때는 검증 구간 데이터가 학습 입력 윈도우에 섞이지 않는지 확인한다.